extract data from data/metadata/autotagging_genre.tsv
find genres and classify them
do a exploratory data analysis

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [8]:
data = []
columns = ['track_id', 'artist_id', 'album_id', 'file_path', 'duration', 'genres']

# Open and parse the file manually
with open('../../data/metadata/autotagging_genre.tsv', 'r', encoding='utf-8') as f:
    next(f)  
    for line in f:
        row = line.strip('\n').split('\t')
        base_info = row[:5]
        genres = row[5:]
        base_info.append(genres)
        data.append(base_info)

# Safely load into a Pandas DataFrame
df_1 = pd.DataFrame(data, columns=columns)
df_1.head(30)


,track_id,artist_id,album_id,file_path,duration,genres
0,track_0000214,artist_000014,album_000031,14/214.mp3,124.6,[genre---punkrock]
1,track_0000215,artist_000014,album_000031,15/215.mp3,151.4,[genre---metal]
2,track_0000216,artist_000014,album_000031,16/216.mp3,234.9,[genre---metal]
3,track_0000217,artist_000014,album_000031,17/217.mp3,127.9,[genre---punkrock]
4,track_0000218,artist_000014,album_000031,18/218.mp3,180.7,[genre---punkrock]
5,track_0000219,artist_000014,album_000031,19/219.mp3,199.4,[genre---metal]
6,track_0000220,artist_000014,album_000031,20/220.mp3,174.3,[genre---punkrock]
7,track_0000221,artist_000014,album_000031,21/221.mp3,224.6,[genre---punkrock]
8,track_0000222,artist_000014,album_000031,22/222.mp3,161.9,[genre---punkrock]
9,track_0000223,artist_000014,album_000031,23/223.mp3,217.9,[genre---metal]


In [9]:
data = []
columns = ['track_id', 'artist_id', 'album_id', 'file_path', 'duration', 'instruments']

# Open and parse the file manually
with open('../../data/metadata/autotagging_instrument.tsv', 'r', encoding='utf-8') as f:
    next(f)  
    for line in f:
        row = line.strip('\n').split('\t')
        base_info = row[:5]
        instruments = row[5:]
        base_info.append(instruments)
        data.append(base_info)

# Safely load into a Pandas DataFrame
df_2 = pd.DataFrame(data, columns=columns)
df_2.head(30)


,track_id,artist_id,album_id,file_path,duration,instruments
0,track_0000382,artist_000020,album_000046,82/382.mp3,211.1,[instrument---voice]
1,track_0000383,artist_000020,album_000046,83/383.mp3,113.1,[instrument---voice]
2,track_0000384,artist_000020,album_000046,84/384.mp3,115.7,[instrument---voice]
3,track_0000386,artist_000020,album_000046,86/386.mp3,103.4,[instrument---voice]
4,track_0000387,artist_000020,album_000046,87/387.mp3,257.1,[instrument---voice]
5,track_0000388,artist_000020,album_000046,88/388.mp3,124.6,[instrument---voice]
6,track_0000389,artist_000020,album_000046,89/389.mp3,150.8,[instrument---voice]
7,track_0000390,artist_000020,album_000046,90/390.mp3,133.7,[instrument---voice]
8,track_0000391,artist_000020,album_000046,91/391.mp3,82.1,[instrument---voice]
9,track_0000392,artist_000020,album_000046,92/392.mp3,127.3,[instrument---voice]


In [10]:
data = []
columns = ['track_id', 'artist_id', 'album_id', 'file_path', 'duration', 'moods']

# Open and parse the file manually
with open('../../data/metadata/autotagging_moodtheme.tsv', 'r', encoding='utf-8') as f:
    next(f)  
    for line in f:
        row = line.strip('\n').split('\t')
        base_info = row[:5]
        moods = row[5:]
        base_info.append(moods)
        data.append(base_info)

# Safely load into a Pandas DataFrame
df_3 = pd.DataFrame(data, columns=columns)
df_3.head(30)


,track_id,artist_id,album_id,file_path,duration,moods
0,track_0000948,artist_000087,album_000149,48/948.mp3,212.7,[mood/theme---background]
1,track_0000950,artist_000087,album_000149,50/950.mp3,248.0,[mood/theme---background]
2,track_0000951,artist_000087,album_000149,51/951.mp3,199.7,[mood/theme---background]
3,track_0002165,artist_000326,album_000347,65/2165.mp3,229.0,[mood/theme---film]
4,track_0002263,artist_000320,album_000366,63/2263.mp3,494.7,[mood/theme---melancholic]
5,track_0003346,artist_000517,album_000521,46/3346.mp3,195.0,"[mood/theme---calm, mood/theme---melodic]"
6,track_0003347,artist_000517,album_000521,47/3347.mp3,201.8,"[mood/theme---calm, mood/theme---melodic]"
7,track_0003348,artist_000517,album_000521,48/3348.mp3,253.3,"[mood/theme---calm, mood/theme---melodic]"
8,track_0003349,artist_000517,album_000521,49/3349.mp3,228.4,"[mood/theme---calm, mood/theme---melodic]"
9,track_0003350,artist_000517,album_000521,50/3350.mp3,194.7,"[mood/theme---calm, mood/theme---melodic]"


In [28]:
# --- 1. Clean prefixes and turn each into a set of tags per track ---
def clean_tags(tag_list, prefix):
    return set(t.replace(prefix, '') for t in tag_list)

df_1['genres']      = df_1['genres'].apply(lambda x: clean_tags(x, 'genre---'))
df_2['instruments']  = df_2['instruments'].apply(lambda x: clean_tags(x, 'instrument---'))
df_3['moods']        = df_3['moods'].apply(lambda x: clean_tags(x, 'mood/theme---'))

# --- 2. Keep only the columns we need for the join ---
g = df_1[['track_id', 'artist_id', 'album_id', 'file_path', 'duration', 'genres']]
i = df_2[['track_id', 'instruments']]
m = df_3[['track_id', 'moods']]

# --- 3. Inner-join: track must have genre AND instrument AND mood/theme tags ---
df_full = g.merge(i, on='track_id', how='inner').merge(m, on='track_id', how='inner')

# Drop rows where any of the tag sets ended up empty (defensive check)
df_full = df_full[
    df_full['genres'].map(len).gt(0) &
    df_full['instruments'].map(len).gt(0) &
    df_full['moods'].map(len).gt(0)
].reset_index(drop=True)

print(f"Tracks with genre + instrument + mood/theme tags: {len(df_full):,}")

# --- 4. Explode genres to count how many qualifying tracks exist per genre ---
df_exploded = df_full.explode('genres')
genre_counts_full = df_exploded['genres'].value_counts()
print(genre_counts_full.head(20))

Tracks with genre + instrument + mood/theme tags: 10,516
genres
soundtrack       2487
electronic       2240
classical        1891
ambient          1859
pop              1616
easylistening    1336
rock             1063
chillout         1019
orchestral        768
folk              689
newage            670
dance             635
experimental      610
house             534
indie             532
popfolk           458
poprock           440
jazz              432
lounge            428
atmospheric       387
Name: count, dtype: int64


In [29]:
df_full.head()

,track_id,artist_id,album_id,file_path,duration,genres,instruments,moods
0,track_0000948,artist_000087,album_000149,48/948.mp3,212.7,"{electronic, lounge, downtempo, easylistening,...",{synthesizer},{background}
1,track_0000950,artist_000087,album_000149,50/950.mp3,248.0,"{electronic, lounge, techno, easylistening, ch...",{synthesizer},{background}
2,track_0000951,artist_000087,album_000149,51/951.mp3,199.7,"{electronic, ambient, lounge, techno, easylist...",{synthesizer},{background}
3,track_0006247,artist_000811,album_000960,47/6247.mp3,381.7,{soundtrack},"{cello, trombone, keyboard}",{emotional}
4,track_0006248,artist_000811,album_000960,48/6248.mp3,249.8,"{classical, soundtrack}",{piano},{documentary}


In [30]:
# --- 5. Pick your 6 genres once you've confirmed counts above ---
target_genres = ['electronic', 'rock', 'pop', 'classical', 'orchestral','jazz']  # adjust based on genre_counts_full

N_PER_GENRE = 750
SEED = 42

sampled_track_ids = []
for genre in target_genres:
    subset_ids = df_exploded.loc[df_exploded['genres'] == genre, 'track_id'].unique()
    n_available = len(subset_ids)
    if n_available < N_PER_GENRE:
        print(f"⚠️  {genre}: only {n_available} available (< {N_PER_GENRE})")
    rng = np.random.default_rng(SEED)
    chosen = rng.choice(subset_ids, size=min(N_PER_GENRE, n_available), replace=False)
    sampled_track_ids.extend(chosen)

# Deduplicate track_ids (a track can be sampled under >1 genre)
sampled_track_ids = list(dict.fromkeys(sampled_track_ids))

# Pull full rows — with original, non-exploded genre sets — from df_full
df_stratified = df_full[df_full['track_id'].isin(sampled_track_ids)].reset_index(drop=True)

print(f"Final stratified dataset: {len(df_stratified):,} unique tracks")

⚠️  jazz: only 432 available (< 750)
Final stratified dataset: 3,798 unique tracks


In [31]:
# --- 6. Serialize tag sets to delimited strings for CSV storage ---
df_csv = df_stratified.copy()

for col in ['genres', 'instruments', 'moods']:
    df_csv[col] = df_csv[col].apply(lambda tags: ','.join(sorted(tags)))

# --- 7. Reorder/select columns for clarity ---
df_csv = df_csv[['track_id', 'artist_id', 'album_id', 'file_path', 'duration', 'genres', 'instruments', 'moods']]

# --- 8. Save to CSV ---
output_path = '../../data/metadata/stratified_sample.csv'
df_csv.to_csv(output_path, index=False)

print(f"Saved {len(df_csv):,} tracks to {output_path}")
df_csv.head()

Saved 3,798 tracks to ../../data/metadata/stratified_sample.csv


,track_id,artist_id,album_id,file_path,duration,genres,instruments,moods
0,track_0006719,artist_000937,album_001020,19/6719.mp3,190.2,"alternative,pop,rock",piano,relaxing
1,track_0006720,artist_000937,album_001020,20/6720.mp3,132.4,pop,piano,relaxing
2,track_0006721,artist_000937,album_001020,21/6721.mp3,151.6,pop,piano,relaxing
3,track_0006723,artist_000937,album_001020,23/6723.mp3,213.8,"minimal,pop",piano,relaxing
4,track_0006724,artist_000937,album_001020,24/6724.mp3,155.5,pop,piano,relaxing


In [ ]:
# 1. Define the 6 target genres
target_genres = {'electronic', 'rock', 'pop', 'classical', 'orchestral', 'jazz'}

# 2. Function to clean and count target genres
def get_target_genres(genre_list):
    cleaned_genres = {g.replace('genre---', '') for g in genre_list}
    return list(cleaned_genres.intersection(target_genres))

# 3. Apply to df_1
df_1['matched_targets'] = df_1['genres'].apply(get_target_genres)
df_1['target_count'] = df_1['matched_targets'].apply(len)


In [ ]:
# --- Split 1: Single genre tracks ---
# Tracks that have exactly 1 total genre, and it is a target genre
df_single = df_1[(df_1['genres'].apply(len) == 1) & (df_1['target_count'] == 1)].copy()

# Extract the single target genre as a string for stratification
df_single['stratify_label'] = df_single['matched_targets'].apply(lambda x: x[0])

# Perform stratified sample
def create_stratified_sample(df, label_col, n_samples=1200, random_state=42):
    min_class_size = df[label_col].value_counts().min()
    n_per_class = min(n_samples // df[label_col].nunique(), min_class_size)
    return df.groupby(label_col, group_keys=False).apply(lambda x: x.sample(n=n_per_class, random_state=random_state))

df_single_sample = create_stratified_sample(df_single, 'stratify_label', n_samples=1200) # Aiming for 200 per genre
print("Split 1 (Single Genre) Sample Distribution:")
print(df_single_sample['stratify_label'].value_counts())

# Save to CSV
import os
os.makedirs('../../data/metadata', exist_ok=True)
df_single_sample.to_csv('../../data/metadata/stratified_single_genre.csv', index=False)


In [ ]:
# --- Split 2: 2 or 3 genres tracks ---
# Tracks with 2 or 3 total genres, containing at least 1 target genre
df_multi = df_1[(df_1['genres'].apply(len).isin([2, 3])) & (df_1['target_count'] > 0)].copy()

# For stratification, we can simplify by stratifying based on the first matched target genre
df_multi['stratify_label'] = df_multi['matched_targets'].apply(lambda x: x[0])

df_multi_sample = create_stratified_sample(df_multi, 'stratify_label', n_samples=1200) # Aiming for 200 per genre
print("\n
Split 2 (2 or 3 Genres) Sample Distribution:")
print(df_multi_sample['stratify_label'].value_counts())

# Save to CSV
df_multi_sample.to_csv('../../data/metadata/stratified_multi_genre.csv', index=False)
